# Setup


In [1]:
%pip install ipywidgets
%pip install duckdb

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
sys.path.append('../scripts')

# Extract the Features


In [ ]:
import os
import features
from log import log


def extract_features(dataset: str = "stackoverflow_dba", version: str = "v1.0", sqlstorm: bool = True):
    os.makedirs(f"features/{version}", exist_ok=True)
    input_file = f"../results/{version}/features/{dataset}{'_sqlstorm_' + version if sqlstorm else ''}.csv"
    output_file = f"features/{version}/{dataset}{'_sqlstorm' if sqlstorm else ''}.csv.gz"

    log.info(f"Extracting features for {dataset} version {version} (SQLStorm: {sqlstorm})")
    with log.file(output_file.replace(".csv.gz", ".log")) as log_file:
        features.compute(input_file, output_file=output_file)

In [4]:
extract_features(dataset="job", version="v0.0", sqlstorm=False)
extract_features(dataset="tpchSf1", version="v0.0", sqlstorm=True)
extract_features(dataset="tpcdsSf1", version="v0.0", sqlstorm=True)

extract_features(dataset="stackoverflow_dba", version="v1.0", sqlstorm=True)
extract_features(dataset="job", version="v1.0", sqlstorm=True)
extract_features(dataset="tpchSf1", version="v1.0", sqlstorm=True)
extract_features(dataset="tpcdsSf1", version="v1.0", sqlstorm=True)

[17:11:55]  INFO        Extracting features for job version v0.0 (SQLStorm:     
                        False)                                                  

            INFO        Extracting features for tpchSf1 version v0.0 (SQLStorm: 
                        True)                                                   

[17:13:04]  INFO        Extracting features for tpcdsSf1 version v0.0           
                        (SQLStorm: True)                                        

[17:16:35]  INFO        Extracting features for stackoverflow_dba version v1.0  
                        (SQLStorm: True)                                        

[17:17:21]  INFO        Extracting features for job version v1.0 (SQLStorm:     
                        True)                                                   

[17:17:48]  INFO        Extracting features for tpchSf1 version v1.0 (SQLStorm: 
                        True)                                                   

[17:18:26]  INFO      

In [ ]:
extract_features(dataset="stackoverflow_dba", version="v1.1", sqlstorm=True)
extract_features(dataset="stackoverflow_dba", version="v1.2", sqlstorm=True)
extract_features(dataset="stackoverflow_dba", version="v1.3", sqlstorm=True)
extract_features(dataset="stackoverflow_dba", version="v1.4", sqlstorm=True)
extract_features(dataset="stackoverflow_dba", version="v1.5", sqlstorm=True)

[14:56:38]  INFO        Extracting features for stackoverflow_dba version v1.1  
                        (SQLStorm: True)                                        

[14:57:00]  INFO        Extracting features for stackoverflow_dba version v1.2  
                        (SQLStorm: True)                                        

            INFO        Extracting features for stackoverflow_dba version v1.3  
                        (SQLStorm: True)                                        

[14:57:01]  INFO        Extracting features for stackoverflow_dba version v1.4  
                        (SQLStorm: True)                                        

[14:57:02]  INFO        Extracting features for stackoverflow_dba version v1.5  
                        (SQLStorm: True)                                        



In [ ]:
extract_features(dataset="stackoverflow_dba", version="v2.0", sqlstorm=True)

[16:53:51]  INFO        Extracting features for stackoverflow_dba version v2.0  
                        (SQLStorm: True)                                        



# Load the Features


In [2]:
import duckdb


def load_features(dataset: str = "stackoverflow_dba", version: str = "v1.0", sqlstorm: bool = True):
    file = f"features/{version}/{dataset}{'_sqlstorm' if sqlstorm else ''}.csv.gz"
    features = duckdb.read_csv(file)
    operators = duckdb.read_csv(file.replace(".csv.gz", "_operators.csv.gz"))
    expressions = duckdb.read_csv(file.replace(".csv.gz", "_expressions.csv.gz"))

    operators2 = duckdb.sql("select query, operator from operators where operator <> 'Result' and operator <> 'GroupJoin' and operator <> 'Map' and operator <> 'IterationScan' and operator <> 'InlineTable'"
                            "union all select query, 'Join' as operator from operators where operator = 'GroupJoin'"
                            "union all select query, 'GroupBy' as operator from operators where operator = 'GroupJoin'")
    expressions2 = duckdb.sql("select query, expression, category, type from expressions where category <> 'base'")

    return features, operators2, expressions2

In [ ]:


job_small, job_small_ops, job_small_exprs = load_features(dataset="job", version="v0.0", sqlstorm=False)
tpch_small, tpch_small_ops, tpch_small_exprs = load_features(dataset="tpchSf1", version="v0.0", sqlstorm=True)
tpcds_small, tpcds_small_ops, tpcds_small_exprs = load_features(dataset="tpcdsSf1", version="v0.0", sqlstorm=True)

so, so_ops, so_exprs = load_features(dataset="stackoverflow_dba", version="v1.0", sqlstorm=True)
job, job_ops, job_exprs = load_features(dataset="job", version="v1.0", sqlstorm=True)
tpch, tpch_ops, tpch_exprs = load_features(dataset="tpchSf1", version="v1.0", sqlstorm=True)
tpcds, tpcds_ops, tpcds_exprs = load_features(dataset="tpcdsSf1", version="v1.0", sqlstorm=True)

# Analysis


In [9]:

result = duckdb.sql("""
           select   complexity, 
                    count(*)::double num_queries,
                    avg(querylength) avg_querylength,
                    avg((select count(*) from so_ops o where o.query = f.query))  as avg_ops,
                    avg((select count(*) from so_ops e where e.query = f.query and operator = 'Join'))  as avg_joins,    
                    avg((select count(*) from so_exprs e where e.query = f.query))  as avg_exprs,           
           from so f
           group by complexity
           union all
           select  'all' complexity, 
                    count(*)::double num_queries,
                    avg(querylength) avg_querylength,
                    avg((select count(*) from so_ops o where o.query = f.query))  as avg_ops,
                    avg((select count(*) from so_ops e where e.query = f.query and operator = 'Join'))  as avg_joins,    
                    avg((select count(*) from so_exprs e where e.query = f.query))  as avg_exprs,
           from so f
           union all
           select 'job_small' complexity,
                    count(*)::double num_queries,
                    avg(querylength) avg_querylength,
                    avg((select count(*) from job_small_ops o where o.query = f.query))  as avg_ops,
                    avg((select count(*) from job_small_ops e where e.query = f.query and operator = 'Join'))  as avg_joins,    
                    avg((select count(*) from job_small_exprs e where e.query = f.query))  as avg_exprs,
           from job_small f
           union all
           select 'tpch_small' complexity,
                    count(*)::double num_queries,
                    avg(querylength) avg_querylength,
                    avg((select count(*) from tpch_small_ops o where o.query = f.query))  as avg_ops,
                    avg((select count(*) from tpch_small_ops e where e.query = f.query and operator = 'Join'))  as avg_joins,    
                    avg((select count(*) from tpch_small_exprs e where e.query = f.query))  as avg_exprs,
           from tpch_small f
           union all
           select 'tpcds_small' complexity,
                    count(*)::double num_queries,
                    avg(querylength) avg_querylength,
                    avg((select count(*) from tpcds_small_ops o where o.query = f.query))  as avg_ops,
                    avg((select count(*) from tpcds_small_ops e where e.query = f.query and operator = 'Join'))  as avg_joins,    
                    avg((select count(*) from tpcds_small_exprs e where e.query = f.query))  as avg_exprs,
            from tpcds_small f
            union all
            select 'job' complexity,
                    count(*)::double num_queries,
                    avg(querylength) avg_querylength,
                    avg((select count(*) from job_ops o where o.query = f.query))  as avg_ops,
                    avg((select count(*) from job_ops e where e.query = f.query and operator = 'Join'))  as avg_joins,    
                    avg((select count(*) from job_exprs e where e.query = f.query))  as avg_exprs,
            from job f
            union all
            select 'tpch' complexity,
                    count(*)::double num_queries,
                    avg(querylength) avg_querylength,
                    avg((select count(*) from tpch_ops o where o.query = f.query))  as avg_ops,
                    avg((select count(*) from tpch_ops e where e.query = f.query and operator = 'Join'))  as avg_joins,    
                    avg((select count(*) from tpch_exprs e where e.query = f.query))  as avg_exprs,
            from tpch f
            union all
            select 'tpcds' complexity,
                    count(*)::double num_queries,
                    avg(querylength) avg_querylength,
                    avg((select count(*) from tpcds_ops o where o.query = f.query))  as avg_ops,
                    avg((select count(*) from tpcds_ops e where e.query = f.query and operator = 'Join'))  as avg_joins,    
                    avg((select count(*) from tpcds_exprs e where e.query = f.query))  as avg_exprs
            from tpcds f
""").show()

┌─────────────┬─────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┐
│ complexity  │ num_queries │  avg_querylength   │      avg_ops       │     avg_joins      │     avg_exprs      │
│   varchar   │   double    │       double       │       double       │       double       │       double       │
├─────────────┼─────────────┼────────────────────┼────────────────────┼────────────────────┼────────────────────┤
│ high        │      3460.0 │ 1347.1583815028903 │  22.07687861271676 │  6.199421965317919 │  35.61387283236994 │
│ medium      │     10195.0 │ 1168.0664051005394 │ 14.407552721922512 │  4.166552231486023 │ 29.123589995095635 │
│ low         │      4596.0 │  333.9264577893821 │  6.422758920800696 │ 1.8748912097476067 │  8.137946040034812 │
│ all         │     18251.0 │  991.9637828064216 │ 13.850747904224425 │ 3.9748506931127063 │ 25.069366062133582 │
│ job_small   │       113.0 │  825.0442477876106 │ 17.292035398230087 │  7.6460176991150

In [10]:
result = duckdb.sql("""
                    with ops as (select distinct operator from so_ops),
                         cops as (select complexity, operator, 
                                        count(*)::double / (select count(*) from so f2 where f2.complexity = f.complexity)::double as c 
                                    from so_ops o, so f where o.query = f.query group by complexity, operator),
                         cops_tpch as (select operator, 
                                        count(*)::double / (select count(*) from tpch_small)::double as c 
                                    from tpch_small_ops o group by operator),
                         cops_tpcds as (select operator, 
                                        count(*)::double / (select count(*) from tpcds_small)::double as c 
                                    from tpcds_small_ops o group by operator)
                    select ops.operator, 
                        (select c from cops c where c.operator = ops.operator and c.complexity = 'low') as low,
                        (select c from cops c where c.operator = ops.operator and c.complexity = 'medium') as medium,
                        (select c from cops c where c.operator = ops.operator and c.complexity = 'high') as high,
                        (select c from cops_tpch c where c.operator = ops.operator) as tpch,
                        (select c from cops_tpcds c where c.operator = ops.operator) as tpcds
                    from ops
                    where ops.operator not in ('PipelineBreakerScan', 'Temp')
                    order by low desc nulls last, medium desc nulls last, high desc nulls last
""")
result.show()

for r in result.fetchall():
    def conv(c):
        return f"0" if c is None else (f"{c:.3f}" if c < 0.01 else f"{c:.2f}")
    print(f"{r[0]} & {conv(r[1])} & {conv(r[2])} & {conv(r[3])} & {conv(r[4])} & {conv(r[5])} \\\\")

┌──────────────┬───────────────────────┬───────────────────────┬───────────────────────┬─────────────────────┬────────────────────┐
│   operator   │          low          │        medium         │         high          │        tpch         │       tpcds        │
│   varchar    │        double         │        double         │        double         │       double        │       double       │
├──────────────┼───────────────────────┼───────────────────────┼───────────────────────┼─────────────────────┼────────────────────┤
│ TableScan    │    2.8729329852045256 │     4.939676311917607 │     5.749132947976879 │  3.6788863636363636 │  7.235436893203883 │
│ Join         │    1.8748912097476067 │     4.166552231486023 │     6.199421965317919 │             2.81525 │  6.307009708737864 │
│ Sort         │    0.9873803307223673 │    1.0277587052476704 │    1.0965317919075144 │  0.8181818181818182 │ 0.8332815533980582 │
│ GroupBy      │    0.6818973020017406 │    2.3321235899950956 │    4.009826

In [11]:
result = duckdb.sql("""
                    with exps as (select distinct category from so_exprs),
                         cexps as (select complexity, category, count(*)::double / (select count(*) from so f2 where f2.complexity = f.complexity)::double as c from so_exprs o, so f where o.query = f.query group by complexity, category),
                            cexps_tpch as (select category, count(*)::double / (select count(*) from tpch_small)::double as c from tpch_small_exprs o group by category),
                            cexps_tpcds as (select category, count(*)::double / (select count(*) from tpcds_small)::double as c from tpcds_small_exprs o group by category)
                    select exps.category, 
                        (select c from cexps c where c.category = exps.category and c.complexity = 'low') as low,
                        (select c from cexps c where c.category = exps.category and c.complexity = 'medium') as medium,
                        (select c from cexps c where c.category = exps.category and c.complexity = 'high') as high,
                        (select c from cexps_tpch c where c.category = exps.category) as tpch,
                        (select c from cexps_tpcds c where c.category = exps.category) as tpcds
                    from exps
                    order by low desc nulls last, medium desc nulls last, high desc nulls last
""")
result.show()

for r in result.fetchall():
    def conv(c):
        return f"0" if c is None else (f"{c:.3f}" if c < 0.01 else f"{c:.2f}")
    print(f"{r[0]} & {conv(r[1])} & {conv(r[2])} & {conv(r[3])} & {conv(r[4])} & {conv(r[5])} \\\\")

┌─────────────────────┬──────────────────────┬──────────────────────┬────────────────────────┬─────────────────────┬──────────────────────┐
│      category       │         low          │        medium        │          high          │        tpch         │        tpcds         │
│       varchar       │        double        │        double        │         double         │       double        │        double        │
├─────────────────────┼──────────────────────┼──────────────────────┼────────────────────────┼─────────────────────┼──────────────────────┤
│ comparison_low      │    4.001087902523934 │   10.753310446297204 │     15.035260115606937 │   7.161681818181818 │   24.849708737864077 │
│ agg_low             │     2.93668407310705 │    8.926630701324179 │      9.128901734104046 │   2.409090909090909 │    4.459398058252427 │
│ cast                │   0.7071366405570061 │    2.758509073075037 │      2.538150289017341 │  0.7727272727272727 │   2.3603495145631066 │
│ case              

In [3]:
sonew, sonew_ops, sonew_exprs = load_features(dataset="stackoverflow_dba", version="v1.1", sqlstorm=True)

In [10]:
result = duckdb.sql("""
with models (model_id, name, input, output) as (values 
    (0, 'gpt-3.5-turbo', 0.250000, 0.750000),
    (1, 'gpt-4o-mini', 0.075000, 0.300000),
    (2, 'gpt-4o', 1.250000, 5.000000),
    (3, 'gpt-4.1-nano', 0.050000, 0.200000),
    (4, 'gpt-4.1-mini', 0.200000, 0.800000),
    (5, 'gpt-4.1', 1.000000, 4.000000),
    (6, 'gpt-5-nano', 0.025000, 0.200000),
    (7, 'gpt-5-mini', 0.125000, 1.000000),
    (8, 'gpt-5', 0.625000, 5.000000),
    (9, 'codex-mini-latest', 1.500000, 6.000000),
    (10, 'nova-micro', 0.017500, 0.070000),
    (11, 'nova-lite', 0.030000, 0.120000),
    (12, 'nova-pro', 0.400000, 1.600000),
    (13, 'nova-premier', 1.250000, 6.250000),
    (14, 'claude-3-haiku', 0.125000, 0.625000),
    (15, 'claude-3.5-haiku', 0.400000, 2.000000),
    (16, 'claude-4.5-sonnet', 1.500000, 7.500000),
    (17, 'claude-4.1-opus', 7.500000, 37.500000),
    (18, 'gemini-2.5-flash-lite', 0.050000, 0.200000),
    (19, 'gemini-2.5-flash', 0.150000, 1.250000),
    (20, 'gemini-2.5-pro', 0.625000, 5.000000),
    (21, 'grok-4-fast-non-reasoning', 0.200000, 0.500000),
    (22, 'grok-code-fast', 0.200000, 1.500000),
    (23, 'grok-4', 3.000000, 15.000000),
    (24, 'gpt-oss-20b', 0.035000, 0.150000),
    (25, 'gpt-oss-120b', 0.075000, 0.300000),
    (26, 'llama-3.3-instruct', 0.360000, 0.360000),
    (27, 'pixtral-large', 2.000000, 6.000000),
    (28, 'deepseek-r1', 1.350000, 5.400000),
    (29, 'qwen3-coder', 0.075000, 0.300000)
),
temp as (select *, replace(query, '.sql', '')::int as id, floor(id / 30000)::int as prompt, floor(floor((id -1) / 1000 ) % 30)::int as mid, name from sonew, models where mid = model_id )
--select state, name, count(*), floor(avg(querylength)) as length, max(id), round(sum(output_tokens *output + input_tokens*input) / 1000000, 3) as price, sum(output_tokens) / count(*) from temp where state = 'success' group by state, prompt, mid, name order by prompt, mid, name;
select * from temp where id between 47000 and 47100 order by id asc
""")
result.show(max_rows=100)

┌───────────┬─────────┬─────────────────────────────────────────────────────────────────────────┬─────────┬───────┬────────────────┬─────────────┬─────────────┬────────┬────────┬────────┬──────────────┬────────┬─────────┬────────────┬────────────────┬────────────────────┬────────────┬─────────────────┬─────────┬─────────────┬───────────┬──────────────┬───────────────┬──────────┬─────────────────┬──────────────┬──────────────┬───────┬──────────┬───────┬─────────────────┐
│   query   │  state  │                                 message                                 │  time   │ rows  │ allocatedBytes │ scannedRows │ querylength │  ops   │ scans  │ joins  │ aggregations │ sorts  │ windows │ iterations │ distinct_trees │ distinct_operators │ complexity │      model      │ prompt  │ temperature │ reasoning │ input_tokens │ output_tokens │ model_id │      name       │    input     │    output    │  id   │ prompt_1 │  mid  │     name_2      │
│  varchar  │ varchar │                           

In [11]:
sqlstorm_v2, _, _ = load_features(dataset="stackoverflow_dba", version="v2.0", sqlstorm=True)
llm = duckdb.read_csv("llm.csv")
queries_v2 = duckdb.read_csv("queries/v2.0/stackoverflow.csv.gz")

duckdb.sql("""
with models as (select model, round(sum(output_tokens*output + input_tokens*input) / 1000000, 3) as price,
            from queries_v2 join llm using (model) group by model),
    features as (select model, count(*) as count, 
           avg(querylength)::int as avg_length,
           max(querylength)::int as max_length,
           avg(ops)::int as avg_ops,
           max(ops)::int as max_ops,
           avg(rows)::int as avg_rows,
           max(rows)::int as max_rows,
           count(*) filter (where rows > 0) / count(*) * 100 as nonempty,
           from sqlstorm_v2 join queries_v2 using (query) group by grouping sets ((model), ()))
select  *, sum(price) over ()
from features f left join models using (model)
order by price asc
""")

┌───────────────────────┬───────┬────────────┬────────────┬─────────┬─────────┬──────────┬──────────┬───────────────────┬────────┬────────────────────┐
│         model         │ count │ avg_length │ max_length │ avg_ops │ max_ops │ avg_rows │ max_rows │     nonempty      │ price  │ sum(price) OVER () │
│        varchar        │ int64 │   int32    │   int32    │  int32  │  int32  │  int32   │  int32   │      double       │ double │       double       │
├───────────────────────┼───────┼────────────┼────────────┼─────────┼─────────┼──────────┼──────────┼───────────────────┼────────┼────────────────────┤
│ nova-micro            │   763 │       1285 │       4726 │      14 │      32 │   119217 │ 36424681 │ 88.33551769331585 │  0.069 │ 25.619999999999997 │
│ gpt-5-nano            │   625 │       2552 │       4698 │      18 │      41 │    12663 │  6131643 │             95.04 │  0.215 │ 25.619999999999997 │
│ gemini-2.5-flash-lite │   683 │       3891 │       9780 │      25 │      76 │    18756

In [ ]:

duckdb.sql("""
select model, ops, rows, time, * from sqlstorm_v2 join queries_v2 using (query) order by ops desc limit 20
""")

┌───────────────────────┬────────┬───────┬─────────┬──────────┬─────────┬─────────┬─────────┬───────┬────────────────┬─────────────┬─────────────┬────────┬────────┬────────┬──────────────┬────────┬─────────┬────────────┬────────────────┬────────────────────┬────────────┬───────────────────────┬─────────┬─────────────┬───────────┬──────────────┬───────────────┬───────┬────────────┬───────┬───────┐
│         model         │  ops   │ rows  │  time   │  query   │  state  │ message │  time   │ rows  │ allocatedBytes │ scannedRows │ querylength │  ops   │ scans  │ joins  │ aggregations │ sorts  │ windows │ iterations │ distinct_trees │ distinct_operators │ complexity │         model         │ prompt  │ temperature │ reasoning │ input_tokens │ output_tokens │ bytes │ characters │ words │ lines │
│        varchar        │ double │ int64 │ double  │ varchar  │ varchar │ varchar │ double  │ int64 │     double     │   double    │    int64    │ double │ double │ double │    double    │ double │ do

In [ ]:
duckdb.sql("""
select model, ops, rows, time, * from sqlstorm_v2 join queries_v2 using (query) order by time desc limit 20
""")

┌───────────────────────┬────────┬──────────┬───────────┬──────────┬─────────┬───────────────────────────────────────────────────────┬───────────┬──────────┬────────────────┬─────────────┬─────────────┬────────┬────────┬────────┬──────────────┬────────┬─────────┬────────────┬────────────────┬────────────────────┬────────────┬───────────────────────┬─────────┬─────────────┬───────────┬──────────────┬───────────────┬───────┬────────────┬───────┬───────┐
│         model         │  ops   │   rows   │   time    │  query   │  state  │                        message                        │   time    │   rows   │ allocatedBytes │ scannedRows │ querylength │  ops   │ scans  │ joins  │ aggregations │ sorts  │ windows │ iterations │ distinct_trees │ distinct_operators │ complexity │         model         │ prompt  │ temperature │ reasoning │ input_tokens │ output_tokens │ bytes │ characters │ words │ lines │
│        varchar        │ double │  int64   │  double   │ varchar  │ varchar │          